# RAGAS 0.3.1+ Korean Testset Generation

**Based on:** [RAGAS Evaluation Framework](../docs/02_research/perplexity_deep_research/02_ragas_evaluation.md)

**Date:** 2025-10-30

---

## Overview

This notebook implements Korean Q/A testset generation using RAGAS 0.3.1+ latest API:

- **Knowledge Graph-based approach** (latest architecture)
- **Korean Persona integration** for diverse query generation
- **Transform-based node processing** for enhanced context
- **Single-hop and multi-hop query support**

### Key Changes from Previous Versions

| Aspect | Old API (deprecated) | New API (0.3.1+) |
|--------|---------------------|------------------|
| Generator | `TestsetGenerator.from_langchain()` | `TestsetGenerator(llm, embeddings, kg, personas)` |
| Generation | `generate_with_langchain_docs()` | `generate(testset_size, query_distribution)` |
| Evolution | `simple, reasoning, multi_context` | `SingleHopSpecificQuerySynthesizer` |
| Critic LLM | Required `critic_llm` parameter | Not used |
| Temperature | Supported | Removed for GPT-5+ models |

## 1. Setup and Imports

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# RAGAS 0.3.1+ imports
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms
from ragas.testset.transforms import HeadlinesExtractor, HeadlineSplitter, KeyphrasesExtractor
from ragas.testset.persona import Persona
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader

print("✅ Imports successful")

## 2. Configuration

In [ ]:
# Load environment variables
load_dotenv()

# Paths
ROOT_DIR = Path.cwd().parent
DOCS_DIR = ROOT_DIR / "data" / "crawled" / "seoul_traffic" / "markdown_deduplicated"
OUTPUT_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_PATH = OUTPUT_DIR / "ragas_korean_testset.csv"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters
TESTSET_SIZE = 50
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"📁 Documents directory: {DOCS_DIR}")
print(f"💾 Output path: {OUTPUT_PATH}")
print(f"🎯 Testset size: {TESTSET_SIZE}")

## 3. Initialize LLM and Embeddings

### Key Point: No Temperature Parameter

GPT-5+ models (including gpt-4o-mini) do not support the `temperature` parameter in RAGAS 0.3.1+.

In [ ]:
# Initialize LLM (no temperature for GPT-5+ models)
base_llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=os.getenv("OPENAI_API_KEY")
)
llm = LangchainLLMWrapper(base_llm)

# Initialize embeddings
base_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=os.getenv("OPENAI_API_KEY")
)
embeddings = LangchainEmbeddingsWrapper(base_embeddings)

print("✅ LLM and embeddings initialized")

## 4. Define Korean Personas

### Critical: Explicit Korean Language Specification

Each persona MUST explicitly state that all questions and answers should be in Korean.

### Persona Design

Based on typical Seoul Metro user segments:

1. **처음 이용자** (First-time user) - Needs detailed guidance
2. **자주 이용하는 승객** (Frequent traveler) - Values efficiency
3. **불만을 가진 승객** (Dissatisfied customer) - Demands immediate resolution
4. **교통약자** (Transportation vulnerable) - Requires accessibility info

In [ ]:
korean_personas = [
    Persona(
        name="지하철 처음 이용자",
        role_description="""
        지하철을 처음 이용하는 승객입니다.
        모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.
        기본적인 이용 방법, 요금, 노선 정보 등에 대해 자세한 안내가 필요합니다.
        질문은 구체적이고 단계별 설명을 요구하는 형태로 작성됩니다.
        예: "지하철 요금은 어떻게 계산되나요?", "환승은 어떻게 하나요?"
        """
    ),
    Persona(
        name="자주 이용하는 승객",
        role_description="""
        지하철을 자주 이용하는 승객입니다.
        모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.
        효율적인 이동, 시간 절약, 편의시설 위치 등 실용적인 정보에 관심이 많습니다.
        질문은 간결하고 핵심적인 정보를 요구하는 형태로 작성됩니다.
        예: "가장 빠른 환승 경로는?", "막차 시간은 언제인가요?"
        """
    ),
    Persona(
        name="불만을 가진 승객",
        role_description="""
        서비스나 시설에 불만을 가진 승객입니다.
        모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.
        문제 상황에 대한 즉각적인 해결책과 보상을 요구합니다.
        질문은 다소 감정적이고 즉각적인 조치를 요구하는 형태로 작성됩니다.
        예: "지연 보상은 어떻게 받나요?", "불편사항을 어디에 신고하나요?"
        """
    ),
    Persona(
        name="교통약자",
        role_description="""
        어르신, 장애인, 임산부 등 교통약자입니다.
        모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.
        접근성, 편의시설, 도움 서비스 등에 대한 정보가 필요합니다.
        질문은 이해하기 쉽고 배려를 요구하는 형태로 작성됩니다.
        예: "휠체어로 이용 가능한가요?", "도움이 필요하면 어디로 연락하나요?"
        """
    )
]

print(f"✅ Created {len(korean_personas)} Korean personas:")
for p in korean_personas:
    print(f"   - {p.name}")

## 5. Load Documents

In [ ]:
print(f"📄 Loading documents from: {DOCS_DIR}")

loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} documents")
print(f"\n📝 Sample document preview:")
print(f"   Content: {documents[0].page_content[:200]}...")
print(f"   Metadata: {documents[0].metadata}")

## 6. Create Knowledge Graph

### Knowledge Graph Approach

RAGAS 0.3.1+ uses Knowledge Graph to:
1. Structure document relationships
2. Enable context-aware query generation
3. Support multi-hop reasoning

In [ ]:
print("🔨 Creating Knowledge Graph...")

kg = KnowledgeGraph()

for doc in documents:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "document_metadata": doc.metadata
            }
        )
    )

print(f"✅ Created Knowledge Graph with {len(kg.nodes)} nodes")

## 7. Apply Transforms

### Transform Pipeline

Transforms enrich the Knowledge Graph by:

1. **HeadlinesExtractor**: Extract section titles
2. **HeadlineSplitter**: Split documents by sections
3. **KeyphrasesExtractor**: Identify key concepts

These transforms enable more targeted and diverse query generation.

In [ ]:
print("🔄 Applying transforms to Knowledge Graph...")

transforms = [
    HeadlinesExtractor(llm=llm, max_num=20),
    HeadlineSplitter(max_tokens=1500),
    KeyphrasesExtractor(llm=llm)
]

apply_transforms(kg, transforms=transforms)

print(f"✅ Transforms applied. Total nodes: {len(kg.nodes)}")
print(f"\n📊 Node increase: {len(documents)} → {len(kg.nodes)} ({len(kg.nodes) - len(documents)} new nodes)")

## 8. Configure Query Synthesizers

### Query Distribution Strategy

- **50% Headlines-based**: Structured questions referencing specific sections
- **50% Keyphrases-based**: Conceptual questions around key themes

This balanced approach ensures diverse query types.

In [ ]:
query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(
            llm=llm,
            property_name="headlines"
        ),
        0.5  # 50% headline-based queries
    ),
    (
        SingleHopSpecificQuerySynthesizer(
            llm=llm,
            property_name="keyphrases"
        ),
        0.5  # 50% keyphrase-based queries
    ),
]

print("✅ Query distribution configured")
print("   - Headlines-based: 50%")
print("   - Keyphrases-based: 50%")

## 9. Generate Korean Testset

### Generation Process

1. Select nodes from Knowledge Graph
2. Apply persona perspective
3. Use synthesizer to generate Korean Q/A pairs
4. Validate and format results

In [ ]:
print(f"\n🎯 Generating Korean testset...")
print(f"   - Target size: {TESTSET_SIZE}")
print(f"   - Personas: {len(korean_personas)}")
print(f"   - Language: Korean (explicitly specified in personas)\n")

# Create generator
generator = TestsetGenerator(
    llm=llm,
    embedding_model=embeddings,
    knowledge_graph=kg,
    persona_list=korean_personas
)

# Generate testset
print("⏳ Generating... (this may take several minutes)\n")

testset = generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution
)

print("\n✅ Generation complete!")

## 10. Convert to DataFrame and Analyze

In [ ]:
# Convert to DataFrame
testset_df = testset.to_pandas()

print(f"✅ Korean testset generated!")
print(f"   - Total samples: {len(testset_df)}")
print(f"   - Columns: {testset_df.columns.tolist()}")

# Display DataFrame info
testset_df.info()

## 11. Sample Korean Q/A Pairs

In [ ]:
print("\n📝 Sample Korean Q/A Pairs:\n")

for idx in range(min(5, len(testset_df))):
    sample = testset_df.iloc[idx]
    print(f"[{idx+1}] Question:")
    print(f"    {sample['user_input'][:150]}...\n")
    print(f"    Answer:")
    print(f"    {sample['reference'][:150]}...\n")
    print(f"    Synthesizer: {sample.get('synthesizer_name', 'N/A')}")
    print("    " + "="*80 + "\n")

## 12. Validate Korean Language Usage

In [ ]:
import re

def contains_korean(text):
    """Check if text contains Korean characters"""
    korean_pattern = re.compile('[ㄱ-ㅎㅏ-ㅣ가-힣]')
    return bool(korean_pattern.search(text))

# Validate Korean usage
korean_questions = testset_df['user_input'].apply(contains_korean).sum()
korean_answers = testset_df['reference'].apply(contains_korean).sum()

print("\n🔍 Korean Language Validation:")
print(f"   - Questions with Korean: {korean_questions}/{len(testset_df)} ({korean_questions/len(testset_df)*100:.1f}%)")
print(f"   - Answers with Korean: {korean_answers}/{len(testset_df)} ({korean_answers/len(testset_df)*100:.1f}%)")

if korean_questions == len(testset_df) and korean_answers == len(testset_df):
    print("\n✅ All Q/A pairs are in Korean!")
else:
    print("\n⚠️  Some Q/A pairs may not be in Korean - check persona definitions")

## 13. Save Results

In [ ]:
# Save to CSV
testset_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f"\n💾 Saved to: {OUTPUT_PATH}")
print(f"   - File size: {OUTPUT_PATH.stat().st_size / 1024:.2f} KB")
print(f"   - Encoding: UTF-8 (Korean support)")

## 14. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"   - Total Q/A pairs: {len(testset_df)}")
print(f"   - Average question length: {testset_df['user_input'].str.len().mean():.1f} chars")
print(f"   - Average answer length: {testset_df['reference'].str.len().mean():.1f} chars")

print(f"\n🎭 Persona Distribution:")
if 'persona' in testset_df.columns:
    print(testset_df['persona'].value_counts())
else:
    print("   (Persona column not available in output)")

print(f"\n🔧 Synthesizer Distribution:")
if 'synthesizer_name' in testset_df.columns:
    print(testset_df['synthesizer_name'].value_counts())

print("\n" + "="*80)
print("✅ Korean RAGAS testset generation complete!")
print("="*80)

---

## Next Steps

1. **Quality Review**: Manually review sample Q/A pairs for quality
2. **Evaluation Setup**: Use this testset for RAG system evaluation
3. **Iteration**: Adjust personas or synthesizers based on results
4. **Scaling**: Generate larger testsets for comprehensive evaluation

## References

- [RAGAS Documentation](https://docs.ragas.io/)
- [RAGAS GitHub](https://github.com/explodinggradients/ragas)
- [RAGAS Evaluation Framework Guide](../docs/02_research/perplexity_deep_research/02_ragas_evaluation.md)